In [58]:
import uproot as ur
import matplotlib.pyplot as plt
import k3d
import numpy as np
import awkward as ak
import json
import random
import math

In [103]:
event_type = "CCDIS"  # Choose between "NCDIS" and "CCDIS"
minQ2 = 1000           # Choose between 100 and 1000

eventPath = f'{event_type}_Q2={minQ2}_Pythia8.json'
input_file = f'RootFiles/pythia8{event_type}_18x275_minQ2={minQ2}_beamEffects_xAngle=-0.025_hiDiv_1.0000.eicrecon.tree.edm4eic.root'

events = ur.open("%s:events"%(input_file))

In [104]:
hit_branches = [key for key in events.keys() if key.endswith("RecHits.position.x")]

cluster_branches = [key for key in events.keys() if key.endswith("Clusters.position.x") and "TruthClusters" not in key]


jetPx = np.array(events['ReconstructedJets/ReconstructedJets.momentum.x'])
jetPy = np.array(events['ReconstructedJets/ReconstructedJets.momentum.y'])
jetPz = np.array(events['ReconstructedJets/ReconstructedJets.momentum.z'])
jetE = np.array(events['ReconstructedJets/ReconstructedJets.energy'])


# Initialize lists to hold filtered arrays
filtered_jetPx = []
filtered_jetPy = []
filtered_jetPz = []
filtered_jetE = []

# Loop through each sub-array in jetPz and apply the mask
for i in range(len(jetPz)):
    mask = jetPz[i] > 0  # Mask for positive values in the current sub-array of jetPz
    
    # Apply the mask to each corresponding sub-array
    filtered_jetPx.append(jetPx[i][mask])
    filtered_jetPy.append(jetPy[i][mask])
    filtered_jetPz.append(jetPz[i][mask])
    filtered_jetE.append(jetE[i][mask])

# Convert the lists back to arrays, if desired
jetPx = np.array(filtered_jetPx, dtype=object)
jetPy = np.array(filtered_jetPy, dtype=object)
jetPz = np.array(filtered_jetPz, dtype=object)
jetE = np.array(filtered_jetE, dtype=object)

In [105]:
tjetPx = np.array(events['GeneratedJets/GeneratedJets.momentum.x'])
tjetPy = np.array(events['GeneratedJets/GeneratedJets.momentum.y'])
tjetPz = np.array(events['GeneratedJets/GeneratedJets.momentum.z'])
tjetE = np.array(events['GeneratedJets/GeneratedJets.energy'])


# Initialize lists to hold filtered arrays
filtered_tjetPx = []
filtered_tjetPy = []
filtered_tjetPz = []
filtered_tjetE = []

# Loop through each sub-array in jetPz and apply the mask
for i in range(len(tjetPz)):
    mask = tjetPz[i] > 0  # Mask for positive values in the current sub-array of jetPz
    
    # Apply the mask to each corresponding sub-array
    filtered_tjetPx.append(tjetPx[i][mask])
    filtered_tjetPy.append(tjetPy[i][mask])
    filtered_tjetPz.append(tjetPz[i][mask])
    filtered_tjetE.append(tjetE[i][mask])

# Convert the lists back to arrays, if desired
tjetPx = np.array(filtered_tjetPx, dtype=object)
tjetPy = np.array(filtered_tjetPy, dtype=object)
tjetPz = np.array(filtered_tjetPz, dtype=object)
tjetE = np.array(filtered_tjetE, dtype=object)

In [106]:
vx = np.array(events['CentralTrackVertices/CentralTrackVertices.position.x'])
vy = np.array(events['CentralTrackVertices/CentralTrackVertices.position.y'])
vz = np.array(events['CentralTrackVertices/CentralTrackVertices.position.z'])
qOverP = np.array(events['CentralCKFTrackParameters/CentralCKFTrackParameters.qOverP'])
Ttheta = np.array(events['CentralCKFTrackParameters/CentralCKFTrackParameters.theta'])
Tphi = np.array(events['CentralCKFTrackParameters/CentralCKFTrackParameters.phi'])

In [107]:
t = []
x = []
y = []
z = []
e = []
Q2 = []
bx = []

def has_negative_value(lst):
    for sublist in lst:
        for value in sublist:
            if value < 0:
                #mint.append(value)
                return True
    return False


for key in hit_branches:
    print(key)
    if "LFHCAL" not in key:
        x.append(np.array(events[key]))
        y.append(np.array(events[key.replace(".position.x", ".position.y")]))
        z.append(np.array(events[key.replace(".position.x", ".position.z")]))
    
        time_key = key.replace(".position.x", ".time")
        if time_key not in events or has_negative_value(np.array(events.get(time_key, []))):
            #print(time_key)
            t.append([np.zeros_like(inner) for inner in np.array(events.get(key, []))])
        else:
            t.append(np.array(events[key.replace(".position.x", ".time")]))
    
        if key.replace(".position.x", ".energy") in events:
            e.append(np.array(events[key.replace(".position.x", ".energy")]))
        elif "DRICH" in key:
            e.append([-2*np.ones_like(inner) for inner in np.array(events[key])])   
        else:
            e.append([-1*np.ones_like(inner) for inner in np.array(events[key])])

x = [[inner for sublist in x for inner in sublist[i]] for i in range(len(x[0]))]
y = [[inner for sublist in y for inner in sublist[i]] for i in range(len(y[0]))]
z = [[inner for sublist in z for inner in sublist[i]] for i in range(len(z[0]))]
t = [[inner for sublist in t for inner in sublist[i]] for i in range(len(t[0]))]
e = [[inner for sublist in e for inner in sublist[i]] for i in range(len(e[0]))]
Q2 = np.array(events["InclusiveKinematicsTruth.Q2"])
bx = np.array(events["InclusiveKinematicsTruth.x"])

B0ECalRecHits/B0ECalRecHits.position.x
B0TrackerRecHits/B0TrackerRecHits.position.x
BackwardMPGDEndcapRecHits/BackwardMPGDEndcapRecHits.position.x
EcalBarrelImagingRecHits/EcalBarrelImagingRecHits.position.x
EcalBarrelScFiRecHits/EcalBarrelScFiRecHits.position.x
EcalEndcapNRecHits/EcalEndcapNRecHits.position.x
EcalEndcapPInsertRecHits/EcalEndcapPInsertRecHits.position.x
EcalEndcapPRecHits/EcalEndcapPRecHits.position.x
EcalFarForwardZDCRecHits/EcalFarForwardZDCRecHits.position.x
EcalLumiSpecRecHits/EcalLumiSpecRecHits.position.x
ForwardMPGDEndcapRecHits/ForwardMPGDEndcapRecHits.position.x
ForwardOffMTrackerRecHits/ForwardOffMTrackerRecHits.position.x
ForwardRomanPotRecHits/ForwardRomanPotRecHits.position.x
HcalBarrelRecHits/HcalBarrelRecHits.position.x
HcalEndcapNRecHits/HcalEndcapNRecHits.position.x
HcalEndcapPInsertRecHits/HcalEndcapPInsertRecHits.position.x
HcalFarForwardZDCRecHits/HcalFarForwardZDCRecHits.position.x
LFHCALRecHits/LFHCALRecHits.position.x
MPGDBarrelRecHits/MPGDBarrel

In [108]:
LFHt = []
LFHx = []
LFHy = []
LFHz = []
LFHe = []

for key in hit_branches:
    if "LFHCAL" in key:
        LFHx.append(np.array(events[key]))
        LFHy.append(np.array(events[key.replace(".position.x", ".position.y")]))
        LFHz.append(np.array(events[key.replace(".position.x", ".position.z")]))
    
        time_key = key.replace(".position.x", ".time")
        if time_key not in events or has_negative_value(np.array(events.get(time_key, []))):
            #print(time_key)
            LFHt.append([np.zeros_like(inner) for inner in np.array(events.get(key, []))])
        else:
            LFHt.append(np.array(events[key.replace(".position.x", ".time")]))
    
        if key.replace(".position.x", ".energy") in events:
            LFHe.append(np.array(events[key.replace(".position.x", ".energy")]))

LFHx = [[inner for sublist in LFHx for inner in sublist[i]] for i in range(len(LFHx[0]))]
LFHy = [[inner for sublist in LFHy for inner in sublist[i]] for i in range(len(LFHy[0]))]
LFHz = [[inner for sublist in LFHz for inner in sublist[i]] for i in range(len(LFHz[0]))]
LFHe = [[inner for sublist in LFHe for inner in sublist[i]] for i in range(len(LFHe[0]))]
LFHt = [[inner for sublist in LFHt for inner in sublist[i]] for i in range(len(LFHt[0]))]

In [109]:
ct = []
cx = []
cy = []
cz = []
ce = []
cc = []

for key in cluster_branches:
    cx.append(np.array(events[key]))
    cy.append(np.array(events[key.replace(".position.x", ".position.y")]))
    cz.append(np.array(events[key.replace(".position.x", ".position.z")]))
    
    time_key = key.replace(".position.x", ".time")

    if time_key not in events or has_negative_value(np.array(events.get(time_key, []))):
        print(time_key)
        ct.append([np.zeros_like(inner) for inner in np.array(events.get(key, []))])
    else:
        ct.append(np.array(events[key.replace(".position.x", ".time")]))
    
    is_hcal = "Hcal" in key or "HCAL" in key
    cc.append([np.full_like(inner, is_hcal, dtype=bool) for inner in np.array(events[key])])
    
    ce.append(np.array(events[key.replace(".position.x", ".energy")]))


# Flatten cc similar to the other lists
cc = [[inner for sublist in cc for inner in sublist[i]] for i in range(len(cc[0]))]
cx = [[inner for sublist in cx for inner in sublist[i]] for i in range(len(cx[0]))]
cy = [[inner for sublist in cy for inner in sublist[i]] for i in range(len(cy[0]))]
cz = [[inner for sublist in cz for inner in sublist[i]] for i in range(len(cz[0]))]
ct = [[inner for sublist in ct for inner in sublist[i]] for i in range(len(ct[0]))]
ce = [[inner for sublist in ce for inner in sublist[i]] for i in range(len(ce[0]))]

In [110]:
# Generate the header section
header = {
    "version": "3.0.0",
    "experiment": f"pythia8 {event_type}_18x275 minQ2 = {minQ2} GeV2",
    "energy_unit": "GeV", # Select "eV", "KeV", "MeV", "GeV", or "TeV"
    "color_bar": "Log", # Select "Lin" or "Log"
    "length_unit": "mm",
    "particles": [
        {
        "angle_rad": [0.0 , 0.0],
        "size": float(150),
        "color_rgba": [float(1), float(0), float(0), float(1)]
        },
        {
        "angle_rad": [3.14159 , 0.0],       
        "size": float(100),
        "color_rgba": [float(0), float(0), float(1), float(1)]
        }
    ],
    "tracker_settings": {
        "B_field_T": float(1.7),
        "tracker_boundary": [float(0.7817*1000), float(-1.235*1000), float(1.88*1000)]
    }
}

# Main data structure
data = {
    "header": header,
    "events": []
}

In [111]:
def calculate_angles(Px, Py, Pz):
    # Calculate magnitude of the momentum vector
    magnitude = np.sqrt(Px**2 + Py**2 + Pz**2)
    
    # Normalize the momentum vector
    Px_norm = Px / magnitude
    Py_norm = Py / magnitude
    Pz_norm = Pz / magnitude
    
    # Calculate theta (angle with respect to z-axis)
    theta = np.arccos(Pz_norm)
    
    # Calculate phi (azimuthal angle in the x-y plane)
    phi = np.arctan2(Py, Px)
    
    return theta, phi

def pseudorapidity_to_polar_ratio(theta):
    
    # Calculate the ratio d(eta)/d(theta)
    ratio = abs(-0.5 * (1 / math.sin(theta / 2)) * (1 / math.cos(theta / 2)))
    
    return ratio

In [112]:
events = []

numEvts = 20
if len(x) < 20:
    numEvts = len(x)

for i in range(numEvts):

    valid_energies = [energy for energy in e[i] + LFHe[i] if energy > 0]
    min_energy = 1
    max_energy = 100
    if valid_energies:
        min_energy = float(min(valid_energies))
        max_energy = float(max(valid_energies))
    info_text = f"Event #{i}"
    if event_type == "NCDIS":
        info_text += f", Q2 = {round(float(Q2[i][0]), 3)} GeV2, x = {round(float(bx[i][0]), 3)}"
        
    event = {
        "event_data": {
            "info_text": info_text,
            "energy_scale": [min_energy, max_energy]
        },
        "hits": [],
        "clusters": [],
        "tracks": [],
        "jets": [],
        "blocks":[]
    }
    
    normalized_energy = [np.log(energy / min_energy) / np.log(max_energy / min_energy) if energy > 0 else energy for energy in e[i]]
    
    hits = []
    for j in range(len(x[i])):      
        energy = float(normalized_energy[j])
        if (energy == -1):
            red, green, blue, alpha = 0, 0, 0, 1
        elif (energy == -2):
            red, green, blue, alpha = 1, 1, 1, 1
        else:
            red, green, blue, alpha = energy, 0, 1-energy, energy
        
        size = 100
        if (energy == -1):
            size = 50
            
        if t[i][j] < 0.1:
            dist = np.sqrt((x[i][j] / 1000) ** 2 + (y[i][j] / 1000) ** 2 + (z[i][j] / 1000) ** 2)
            c = 0.299792
            time = dist / c
        else:
            time = t[i][j]
            
        hit = {
                "position":[float(x[i][j]), float(y[i][j]), float(z[i][j])],
                "time_ns": float(time),
                "size": float(size),
                "color_rgba": [float(red), float(green), float(blue), float(alpha)]
            }
        hits.append(hit)
    event["hits"] = hits
    
    tracks = []
    if len(vx[i]) > 0:
        evx = float(vx[i][0])
        evy = float(vy[i][0])
        evz = float(vz[i][0])
    else:
        evx = float(0)
        evy = float(0)
        evz = float(0)
        
    for j in range(len(qOverP[i])):
        track = {
            "qOverP": float(qOverP[i][j]),
            "angle_rad": [float(Ttheta[i][j]), float(Tphi[i][j])],
            "vertex": [evx, evy, evz],
            "duration_ns": [float(0), float(1000)],  
            "color_rgba": [float(0), float(1), float(1), float(1)]
        }
        tracks.append(track)

    event["tracks"] = tracks

    clusters = []
    for j in range(len(cx[i])):
        
        energy = float(100*ce[i][j])
        is_hcal = cc[i][j]  # This is now properly indexed

        # Set color based on whether the cluster is from Hcal
        red, green, blue, alpha = (0, 1, 1, 0.5) if is_hcal else (0, 1, 0, 0.5)
        
        cluster = {
                "position":[float(cx[i][j]), float(cy[i][j]), float(cz[i][j])],
                "time_ns": float(ct[i][j]),
                "granularity": float(100),
                "length": energy,
                "color_rgba": [float(red), float(green), float(blue), float(alpha)]
            }
        if(energy > 100):
            clusters.append(cluster)
    event["clusters"] = clusters
    
    jets = []
    for j in range(len(jetPx[i])):
        
        energy = float(80*jetE[i][j])
        theta, phi = calculate_angles(jetPx[i][j], jetPy[i][j], jetPz[i][j])
        red, green, blue, alpha = 0, 0.3, 1, 0.5
        op_angle = 0.5 / pseudorapidity_to_polar_ratio(theta)
        dist = float(60*jetE[i][j]/ 1000)
        c = 0.299792
        time = dist / c
    
        jet = {
                "length": energy,
                "R_rad": float(op_angle),
                "angle_rad": [float(theta), float(phi)],
                "color_rgba": [float(red), float(green), float(blue), float(alpha)],
                "time_ns": float(time)
            }
        jets.append(jet)
        
    event["jets"] = jets
    
    LFHnormalized_energy = [np.log(energy / min_energy) / np.log(max_energy / min_energy) if energy > 0 else energy for energy in LFHe[i]]
    blocks = []
    for j in range(len(LFHx[i])):
        
        energy = float(LFHnormalized_energy[j])
        time = t[i][j]
        red, green, blue, alpha = energy, 0, 1-energy, energy
        dist = np.sqrt((LFHx[i][j] / 1000) ** 2 + (LFHy[i][j] / 1000) ** 2 + (LFHz[i][j] / 1000) ** 2)
        c = 0.299792
        time = dist / c
            
        block = {
                "position":[float(LFHx[i][j]), float(LFHy[i][j]), float(LFHz[i][j])],
                "time_ns": float(time),
                "size": [float(50), float(50), float(1400/7)],
                "color_rgba": [float(red), float(green), float(blue), float(alpha)]
            }
        blocks.append(block)
        
    event["blocks"] = blocks
    
        
    events.append(event)
    
data["events"] = events
    

In [113]:
eventPath = f'{event_type}_Q2={minQ2}_Pythia8_20.json'
with open(eventPath, 'w') as outfile:
    json.dump(data, outfile, indent=4)